# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')  # pulled from Colab Secrets, never printed or stored in the file

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("Connected.")

Connected.


In [2]:
import os

# Clone only if not already present (safe to re-run without erroring)
if not os.path.exists('/content/ML-intern-starter'):
    !git clone https://github.com/Khuld13/ML-intern-starter.git

%cd /content/ML-intern-starter
!pwd

# --- HF connection ---
from google.colab import userdata
import duckdb
import pandas as pd

hf_token = userdata.get('HF_TOKEN')  # pulled from Colab Secrets, never printed or stored in the file

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("Connected.")

Cloning into 'ML-intern-starter'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 181 (delta 83), reused 88 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (181/181), 1.90 MiB | 4.02 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/content/ML-intern-starter
/content/ML-intern-starter
Connected.


In [3]:
con.sql("SHOW TABLES").df()

,name


In [4]:
con.sql("""
    CREATE OR REPLACE VIEW dim_content AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""")
con.sql("""
    CREATE OR REPLACE VIEW fact_march AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""")
print("Views ready.")

Views ready.


In [5]:
con.sql("SHOW TABLES").df()

,name
0,dim_content
1,fact_march


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Distributions of key fields
# gsc_impressions, avg_position -> fact_march (daily performance, dev view)
# days_since_update -> dim_content (July 2026 snapshot)

# Get ref_date first (max report_date in March partition)
ref_date = con.sql("SELECT MAX(report_date) AS d FROM fact_march").df()['d'][0]
print("Reference date:", ref_date)

dist_query = f"""
SELECT 'gsc_impressions' AS field,
    MIN(gsc_impressions) AS min_val,
    approx_quantile(gsc_impressions, 0.25) AS p25,
    approx_quantile(gsc_impressions, 0.50) AS median,
    approx_quantile(gsc_impressions, 0.75) AS p75,
    approx_quantile(gsc_impressions, 0.95) AS p95,
    approx_quantile(gsc_impressions, 0.99) AS p99,
    MAX(gsc_impressions) AS max_val,
    AVG(gsc_impressions) AS mean_val
FROM fact_march
WHERE gsc_data_available = TRUE

UNION ALL

SELECT 'gsc_avg_position',
    MIN(gsc_avg_position),
    approx_quantile(gsc_avg_position, 0.25),
    approx_quantile(gsc_avg_position, 0.50),
    approx_quantile(gsc_avg_position, 0.75),
    approx_quantile(gsc_avg_position, 0.95),
    approx_quantile(gsc_avg_position, 0.99),
    MAX(gsc_avg_position),
    AVG(gsc_avg_position)
FROM fact_march
WHERE gsc_data_available = TRUE

UNION ALL

SELECT 'days_since_update',
    MIN(DATE '{ref_date}' - content_updated_date),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.25),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.50),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.75),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.95),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.99),
    MAX(DATE '{ref_date}' - content_updated_date),
    AVG(DATE '{ref_date}' - content_updated_date)
FROM dim_content
"""

con.sql(dist_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Reference date: 2026-03-31 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,field,min_val,p25,median,p75,p95,p99,max_val,mean_val
0,gsc_impressions,1.0,4.000000,16.000000,62.000000,337.000000,950.000000,40084.0,77.721642
1,gsc_avg_position,0.0,3.738511,7.483744,20.192757,62.605234,88.845775,498.0,15.826651
2,days_since_update,-97.0,-62.000000,-50.000000,34.000000,492.000000,508.000000,519.0,34.148043


In [7]:
negative_check = con.sql(f"""
    SELECT
        CASE
            WHEN DATE '{ref_date}' - content_updated_date < 0 THEN 'negative (updated after ref_date)'
            WHEN DATE '{ref_date}' - content_updated_date >= 0 THEN 'non-negative (updated on/before ref_date)'
            ELSE 'null'
        END AS bucket,
        COUNT(*) AS n_rows,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM dim_content
    GROUP BY bucket
    ORDER BY bucket
""").df()

negative_check

,bucket,n_rows,pct
0,negative (updated after ref_date),382739,73.7
1,non-negative (updated on/before ref_date),136867,26.3


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**gsc_impressions** (fact_march, gsc_data_available = TRUE): heavily right-skewed.
Median is 16, but the mean (77.7) sits far above it — p99 is 949 and the max is
40,084, so a small number of very high-traffic pages are pulling the average up.
This is exactly why a flat threshold like "impressions >= 250" matters as a noise
filter (Signal 2) rather than a soft suggestion — without it, low-volume pages sit
in the same pool as outliers 100x their size.

**gsc_avg_position**: milder right skew — median 7.5 vs mean 15.8, max 498. Most
content sits in reasonable ranking territory (p75 = 20.2), but the tail includes
positions that likely reflect near-invisible pages or possible data artifacts
worth a second look before using this field in Signal 3.

**days_since_update** (dim_content, relative to ref_date = 2026-03-31): the field
is unusable as a direct staleness measure. 73.7% of rows (382,739 of 519,606) are
negative — their `content_updated_date` falls after the March reference date.
This isn't a data error or a real "refresh problem": `dim_content` is a single
July 2026 export, so `content_updated_date` reflects one frozen snapshot value per
page, not a running edit history. Comparing that snapshot date against an earlier
reference point (March 31) produces negative values for any page whose one
recorded edit happened between April and July. This is the same limitation
behind the OPPOSITE verdict on the staleness signal in the baseline notebook —
`dim_content` cannot support genuine time-windowed staleness analysis.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal Test 1: Staleness vs decline rate
# Uses monthly_compare logic (Feb -> March impressions) joined to staleness
# Only valid (non-negative) days_since_update rows, per the snapshot limitation found in Section 1

staleness_test = con.sql(f"""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    staleness AS (
        SELECT
            content_hash_id,
            DATE '{ref_date}' - content_updated_date AS days_since_update
        FROM dim_content
        WHERE DATE '{ref_date}' - content_updated_date >= 0
    ),
    bucketed AS (
        SELECT
            s.content_hash_id,
            CASE
                WHEN s.days_since_update < 30 THEN '0-29 days'
                WHEN s.days_since_update < 90 THEN '30-89 days'
                WHEN s.days_since_update < 180 THEN '90-179 days'
                ELSE '180+ days'
            END AS staleness_bucket,
            m.impressions_march
        FROM staleness s
        JOIN march_agg m USING (content_hash_id)
    )
    SELECT
        staleness_bucket,
        COUNT(*) AS n_pages,
        ROUND(AVG(impressions_march), 1) AS avg_impressions_march
    FROM bucketed
    GROUP BY staleness_bucket
    ORDER BY staleness_bucket
""").df()

staleness_test

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n_pages,avg_impressions_march
0,0-29 days,674,145.9
1,180+ days,261,66.7
2,30-89 days,25696,1319.6
3,90-179 days,1325,359.5


In [9]:
con.sql("""
    CREATE OR REPLACE VIEW fact_feb AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')
""")
print("fact_feb ready.")

fact_feb ready.


In [10]:
staleness_decline_test = con.sql(f"""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    staleness AS (
        SELECT
            content_hash_id,
            DATE '{ref_date}' - content_updated_date AS days_since_update
        FROM dim_content
        WHERE DATE '{ref_date}' - content_updated_date >= 0
    ),
    bucketed AS (
        SELECT
            s.content_hash_id,
            CASE
                WHEN s.days_since_update < 30 THEN '0-29 days'
                WHEN s.days_since_update < 90 THEN '30-89 days'
                WHEN s.days_since_update < 180 THEN '90-179 days'
                ELSE '180+ days'
            END AS staleness_bucket,
            m.impressions_march,
            f.impressions_feb,
            CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag
        FROM staleness s
        JOIN march_agg m USING (content_hash_id)
        JOIN feb_agg f USING (content_hash_id)
    )
    SELECT
        staleness_bucket,
        COUNT(*) AS n_pages,
        ROUND(100.0 * AVG(declined_flag), 1) AS pct_declined,
        ROUND(MEDIAN(impressions_march), 1) AS median_impressions_march
    FROM bucketed
    GROUP BY staleness_bucket
    ORDER BY staleness_bucket
""").df()

staleness_decline_test

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n_pages,pct_declined,median_impressions_march
0,0-29 days,642,53.7,8.0
1,180+ days,132,34.1,7.0
2,30-89 days,25471,36.2,338.0
3,90-179 days,902,27.7,6.0


In [11]:
staleness_decline_filtered = con.sql(f"""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    staleness AS (
        SELECT
            content_hash_id,
            DATE '{ref_date}' - content_updated_date AS days_since_update
        FROM dim_content
        WHERE DATE '{ref_date}' - content_updated_date >= 0
    ),
    bucketed AS (
        SELECT
            s.content_hash_id,
            CASE
                WHEN s.days_since_update < 30 THEN '0-29 days'
                WHEN s.days_since_update < 90 THEN '30-89 days'
                WHEN s.days_since_update < 180 THEN '90-179 days'
                ELSE '180+ days'
            END AS staleness_bucket,
            m.impressions_march,
            f.impressions_feb,
            CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag
        FROM staleness s
        JOIN march_agg m USING (content_hash_id)
        JOIN feb_agg f USING (content_hash_id)
        WHERE m.impressions_march >= 250   -- Signal 2 noise floor applied here
    )
    SELECT
        staleness_bucket,
        COUNT(*) AS n_pages,
        ROUND(100.0 * AVG(declined_flag), 1) AS pct_declined,
        ROUND(MEDIAN(impressions_march), 1) AS median_impressions_march
    FROM bucketed
    GROUP BY staleness_bucket
    ORDER BY staleness_bucket
""").df()

staleness_decline_filtered

,staleness_bucket,n_pages,pct_declined,median_impressions_march
0,0-29 days,55,7.3,553.0
1,180+ days,8,0.0,571.0
2,30-89 days,14221,28.8,995.0
3,90-179 days,63,0.0,3676.0


## 2. Signal test #1

### Signal 1: Staleness (days_since_update) vs decline rate

**Assumption tested:** FlyRank's real flag `stale_visible_page` assumes older content
(higher days_since_update) declines more. Tested by bucketing valid-staleness pages
(days_since_update >= 0, per Section 1's snapshot finding) into staleness ranges and
comparing decline rate (Feb -> March impressions) per bucket.

**Unfiltered result:** 0-29 days: 53.7% declined (n=642) | 30-89 days: 36.2% (n=25,471)
| 90-179 days: 27.7% (n=902) | 180+ days: 34.1% (n=132). No clean trend, and every
bucket except 30-89 days had a median impression count of 6-8 — well below the
250-impression noise floor validated in Signal 2.

**Filtered result** (impressions_march >= 250, per Signal 2): 0-29 days: 7.3% (n=55)
| 30-89 days: 28.8% (n=14,221) | 90-179 days: 0.0% (n=63) | 180+ days: 0.0% (n=8).
Applying the volume filter collapsed three of four buckets to sample sizes too small
to trust (8-63 pages) — their 0.0-7.3% readings reflect insufficient data, not a real
absence of decline. Only the 30-89 day bucket retains enough volume (n=14,221) to be
statistically meaningful.

**Verdict: MIXED / INCONCLUSIVE — insufficient sample size.**

This is not the same conclusion the unfiltered exploratory pass suggested (which read
as OPPOSITE — freshest content showing the highest apparent decline). That earlier
read doesn't survive the same noise filter validated in Signal 2. The real limitation
is structural: after restricting to valid, non-negative staleness values and a fair
impression floor, almost all usable content clusters into a single 60-day staleness
window (30-89 days). There is currently no way to fairly compare decline rates across
a wide range of staleness values in this dataset — not because staleness doesn't
matter, but because the valid-staleness population isn't spread out enough to test it.

In [12]:
volume_test = con.sql("""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march WHERE gsc_data_available = TRUE GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb WHERE gsc_data_available = TRUE GROUP BY content_hash_id
    ),
    joined AS (
        SELECT m.content_hash_id, m.impressions_march,
            CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag,
            CASE WHEN m.impressions_march >= 250 THEN '>=250 impressions' ELSE '<250 impressions' END AS volume_bucket
        FROM march_agg m JOIN feb_agg f USING (content_hash_id)
    )
    SELECT volume_bucket, COUNT(*) AS n_pages, ROUND(100.0*AVG(declined_flag),1) AS pct_declined
    FROM joined GROUP BY volume_bucket
""").df()
volume_test

,volume_bucket,n_pages,pct_declined
0,>=250 impressions,68581,22.8
1,<250 impressions,65657,36.2


### Signal 2: Volume (gsc_impressions) as a noise filter

**Assumption tested:** low-impression pages produce inflated apparent decline rates,
making a minimum-volume floor necessary before trusting any decline signal. Tested by
comparing decline rate above/below a 250-impression threshold.

**Result:** >=250 impressions: 22.8% declined (n=68,581) | <250 impressions: 36.2%
declined (n=65,657). Low-volume pages decline at roughly 1.6x the rate of
high-volume pages — a 13.4 percentage point gap on a near-even sample split.

**Verdict: CONFIRMED.** Low-volume pages show meaningfully inflated apparent decline
rates, consistent with small impression counts swinging on noise rather than real
demand change. The 250-impression floor is validated as necessary and is applied
throughout this notebook (Signal 1's filtered test) and in the baseline scoring rule
(ML-07).

In [13]:
position_test = con.sql("""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM fact_march WHERE gsc_data_available = TRUE GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb WHERE gsc_data_available = TRUE GROUP BY content_hash_id
    ),
    joined AS (
        SELECT
            m.content_hash_id, m.impressions_march, m.avg_position_march,
            CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag,
            CASE
                WHEN m.avg_position_march <= 3 THEN '1-3 (top)'
                WHEN m.avg_position_march <= 10 THEN '4-10 (page one)'
                WHEN m.avg_position_march <= 20 THEN '11-20 (page two)'
                ELSE '21+ (deep)'
            END AS position_bucket
        FROM march_agg m
        JOIN feb_agg f USING (content_hash_id)
        WHERE m.impressions_march >= 250
    )
    SELECT position_bucket, COUNT(*) AS n_pages, ROUND(100.0*AVG(declined_flag),1) AS pct_declined
    FROM joined GROUP BY position_bucket
    ORDER BY position_bucket
""").df()
position_test

,position_bucket,n_pages,pct_declined
0,1-3 (top),6209,25.5
1,11-20 (page two),14560,24.7
2,21+ (deep),14030,14.2
3,4-10 (page one),33782,25.1


In [16]:
decay_risk_test = con.sql(f"""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM fact_march WHERE gsc_data_available = TRUE GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb WHERE gsc_data_available = TRUE GROUP BY content_hash_id
    ),
    age AS (
        SELECT content_hash_id,
            DATE '{ref_date}' - content_created_date AS content_age_days
        FROM dim_content
        WHERE DATE '{ref_date}' - content_created_date >= 0
    ),
    joined AS (
        SELECT
            m.content_hash_id, m.impressions_march, m.avg_position_march, a.content_age_days,
            CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag,
            CASE
                WHEN m.avg_position_march <= 10 AND a.content_age_days >= 180
                    THEN 'page_one_decay_risk (matches flag)'
                ELSE 'does not match flag'
            END AS flag_bucket
        FROM march_agg m
        JOIN feb_agg f USING (content_hash_id)
        JOIN age a USING (content_hash_id)
        WHERE m.impressions_march >= 250
    )
    SELECT flag_bucket, COUNT(*) AS n_pages, ROUND(100.0*AVG(declined_flag),1) AS pct_declined
    FROM joined GROUP BY flag_bucket
""").df()
decay_risk_test

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,flag_bucket,n_pages,pct_declined
0,does not match flag,44830,18.9
1,page_one_decay_risk (matches flag),23751,30.2


### Signal 3: Position + age vs page_one_decay_risk

**Assumption tested:** FlyRank's real flag `page_one_decay_risk` (avg_position <= 10
AND content_age_days >= 180) assumes old, well-ranked content is a decay risk —
implying it should show higher decline than content that doesn't match this profile.
Tested directly against the flag's own definition, with the 250-impression floor
(Signal 2) applied throughout and content_age_days computed from content_created_date
(same July-2026-snapshot caveat as Section 1's days_since_update).

**Result:** pages matching the flag definition: 30.2% declined (n=23,751) | pages not
matching: 18.9% declined (n=44,830). Matching pages decline at roughly 1.6x the rate
of non-matching pages — a well-powered, consistent gap.

**Verdict: CONFIRMED.** Unlike Signal 1, this flag's assumption holds: content that is
both well-ranked and old genuinely does decline more than content that isn't. Position
alone (tested earlier, un-adjusted for age) showed a flatter pattern across position
tiers 1-20; adding the age requirement from the real flag definition is what surfaces
the effect. This suggests `page_one_decay_risk`'s two-condition design is doing real
work — position or age alone would likely be weaker signals than the combination.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale_flag_test = con.sql(f"""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march WHERE gsc_data_available = TRUE GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb WHERE gsc_data_available = TRUE GROUP BY content_hash_id
    ),
    staleness AS (
        SELECT content_hash_id,
            DATE '{ref_date}' - content_updated_date AS days_since_update
        FROM dim_content
        WHERE DATE '{ref_date}' - content_updated_date >= 0
    ),
    joined AS (
        SELECT
            m.content_hash_id, m.impressions_march, s.days_since_update,
            CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag,
            CASE
                WHEN s.days_since_update >= 180 AND m.impressions_march >= 500
                    THEN 'stale_visible_page (matches flag)'
                ELSE 'does not match flag'
            END AS flag_bucket
        FROM march_agg m
        JOIN feb_agg f USING (content_hash_id)
        JOIN staleness s USING (content_hash_id)
    )
    SELECT flag_bucket, COUNT(*) AS n_pages, ROUND(100.0*AVG(declined_flag),1) AS pct_declined
    FROM joined GROUP BY flag_bucket
""").df()
stale_flag_test

,flag_bucket,n_pages,pct_declined
0,does not match flag,27142,36.3
1,stale_visible_page (matches flag),5,0.0


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag tested:** `stale_visible_page` (days_since_last_update >= 180 AND impressions_90d
>= 500). "Matches flag" = pages satisfying both conditions simultaneously — untouched
for 6+ months, still drawing meaningful traffic. "Does not match flag" = everything
else (recently updated, or low-traffic, or both).

**Result:** matches flag: 0.0% declined (n=5) | does not match flag: 36.3% declined
(n=27,142).

**Verdict: FALSE — the flag cannot be meaningfully tested on this dataset.** Not
because the direction is wrong, but because only 5 of ~72,000 content items with valid
staleness data satisfy both conditions at once. This directly follows from Signal 1:
73.7% of dim_content rows have unusable (negative) staleness values due to the July
2026 snapshot limitation, and the vast majority of the remaining valid rows cluster in
a 30-89 day window — far short of the 180-day bar this flag requires. Combined with a
500-impression floor, the flag's two conditions almost never co-occur in this snapshot.
The practical implication: as currently defined, `stale_visible_page` would surface
essentially zero candidates if run against this data — not because stale, visible
content doesn't exist, but because the snapshot's staleness field can't reliably
identify it.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The volume floor (>=250 impressions) and the combined position+age rule
(`page_one_decay_risk`) are trustworthy prioritization signals and should stay in any
refresh queue — both showed a consistent, well-powered ~1.6x gap in decline rate.
Staleness alone, however, is not currently usable: `dim_content`'s snapshot design
means most content has no valid staleness value, and FlyRank's own `stale_visible_page`
flag — which requires 180+ days stale AND 500+ impressions — matched only 5 pages out
of ~72,000, meaning it would surface almost no candidates if run today. A content team
relying on that flag alone would believe they have few stale, visible pages needing
review, when the real answer is that the flag can't see most of the inventory well
enough to judge.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Supporting numbers for the practical-implications summary
# Pulls the key stats referenced in the markdown above, in one place for a reviewer to check.

summary = {
    "Signal 2 (volume floor) — decline rate, >=250 impressions": "22.8% (n=68,581)",
    "Signal 2 (volume floor) — decline rate, <250 impressions": "36.2% (n=65,657)",
    "Signal 3 (page_one_decay_risk) — decline rate, matches flag": "30.2% (n=23,751)",
    "Signal 3 (page_one_decay_risk) — decline rate, does not match": "18.9% (n=44,830)",
    "stale_visible_page flag — pages matching definition": "5 (out of ~72,000 valid-staleness pages)",
    "dim_content rows with unusable (negative) staleness": "73.7% (382,739 / 519,606)",
}

for k, v in summary.items():
    print(f"{k}: {v}")

Signal 2 (volume floor) — decline rate, >=250 impressions: 22.8% (n=68,581)
Signal 2 (volume floor) — decline rate, <250 impressions: 36.2% (n=65,657)
Signal 3 (page_one_decay_risk) — decline rate, matches flag: 30.2% (n=23,751)
Signal 3 (page_one_decay_risk) — decline rate, does not match: 18.9% (n=44,830)
stale_visible_page flag — pages matching definition: 5 (out of ~72,000 valid-staleness pages)
dim_content rows with unusable (negative) staleness: 73.7% (382,739 / 519,606)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.